# Scrapy untuk Kasus Nyata (Real Case)

Notebook lain memakai *sandbox* (`quotes.toscrape`, `books.toscrape`). Di sini kita lihat
**bagaimana menghadapi e-commerce sungguhan** — pakai contoh URL nyata:

> `https://www.blibli.com/c3/olahraga-sepeda/AK-1000051`

Pelajaran utamanya: **scraping kasus nyata itu 70% riset, 30% kode.** Sebelum menulis spider,
kita wajib menjawab dua pertanyaan:

1. **Boleh tidak?** → cek `robots.txt` dan Terms of Service.
2. **Bisa tidak?** → halaman statis (HTML) atau SPA (JavaScript + API)? Ada bot-protection?

Kita kerjakan recon-nya di Blibli, ambil keputusan, lalu bangun **spider Scrapy lengkap yang
benar-benar jalan** di target yang diizinkan dengan pola yang sama persis.

## 1. Recon Blibli — "Boleh tidak?" (cek `robots.txt`)

`robots.txt` adalah aturan main yang dipublikasikan situs: path mana yang boleh/tidak boleh
diakses bot. Kita pakai `urllib.robotparser` untuk mengeceknya secara terprogram —
persis yang dilakukan Scrapy saat `ROBOTSTXT_OBEY = True`.

In [1]:
import requests
import urllib.robotparser as urobot
from protego import Protego  # parser robots.txt yang dipakai Scrapy

BROWSER_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
              "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36")

# PENTING: ambil robots.txt dengan UA browser. Kalau pakai UA default Python,
# Blibli sering balas 403 -> parser menganggap "semua DILARANG" (hasil menyesatkan).
robots = requests.get("https://www.blibli.com/robots.txt",
                      headers={"User-Agent": BROWSER_UA}, timeout=20).text

rp = urobot.RobotFileParser()
rp.parse(robots.splitlines())   # parser bawaan Python
pr = Protego.parse(robots)      # parser Scrapy (lebih akurat soal wildcard)

paths = {
    "Halaman kategori (yang kamu buka)": "/c3/olahraga-sepeda/AK-1000051",
    "API produk (pengisi halaman)":      "/backend/search/products",
    "Halaman detail produk":             "/p/nama-produk/ps--ABC-12345",
}
print(f"{'PATH':36} | urllib | Protego(Scrapy)")
print("-" * 66)
for label, path in paths.items():
    a = "BOLEH" if rp.can_fetch("*", path) else "DILARANG"
    b = "BOLEH" if pr.can_fetch(path, "*") else "DILARANG"
    print(f"{label:36} | {a:6} | {b}")

PATH                                 | urllib | Protego(Scrapy)
------------------------------------------------------------------
Halaman kategori (yang kamu buka)    | BOLEH  | BOLEH
API produk (pengisi halaman)         | BOLEH  | DILARANG
Halaman detail produk                | BOLEH  | BOLEH


Dua pelajaran dari tabel di atas:

1. **API `/backend/search/*` DILARANG** — padahal endpoint itulah yang mengisi daftar produk
   pada halaman kategori. Jadi **sumber data utamanya tidak boleh** diambil. (Halaman kategori
   `/c3/...` dan detail `/p/...` sendiri `Allow`, tapi isinya datang dari API yang dilarang.)
2. **`urllib.robotparser` dan Protego bisa beda jawaban.** `urllib` keliru bilang API "BOLEH"
   karena lemah menangani wildcard `*`; **Protego — yang dipakai Scrapy — benar bilang
   DILARANG.** Inilah kenapa kepatuhan robots sebaiknya diserahkan ke Scrapy
   (`ROBOTSTXT_OBEY=True`), bukan parser seadanya.

## 2. Recon Blibli — "Bisa tidak?" (statis vs SPA + bot-protection)

Sekarang kita cek apakah data produk ada di HTML mentah (statis) atau halaman ini SPA yang
butuh JavaScript, sekaligus melihat apakah ada proteksi anti-bot.

In [2]:
URL = "https://www.blibli.com/c3/olahraga-sepeda/AK-1000051"
try:
    r = requests.get(URL, headers={"User-Agent": BROWSER_UA}, timeout=20)
    html = r.text
    print("HTTP status :", r.status_code, "(200=OK, 403=diblokir bot-protection)")
    print("Ukuran HTML :", len(html), "byte\n")

    n_state = html.count("__INITIAL_STATE__") + html.count("window.__")
    n_nama  = html.count("olahraga-sepeda")
    print(f"Penanda SPA (state JS)        : {n_state}x  <- data ditaruh di blob JS, bukan tag HTML")
    print(f"Link/teks 'olahraga-sepeda'   : {n_nama}x  <- nama produk kategori ini tak ada di HTML statis")

    if r.status_code == 403:
        print("\n=> Diblokir (403): klien non-browser ditolak proteksi anti-bot.")
    else:
        print("\n=> Halaman SPA: HTML kerangka saja; daftar produk diisi via API yang DILARANG robots.")
except Exception as e:
    print("Gagal mengambil halaman:", e)

HTTP status : 200 (200=OK, 403=diblokir bot-protection)
Ukuran HTML : 186207 byte

Penanda SPA (state JS)        : 6x  <- data ditaruh di blob JS, bukan tag HTML
Link/teks 'olahraga-sepeda'   : 0x  <- nama produk kategori ini tak ada di HTML statis

=> Halaman SPA: HTML kerangka saja; daftar produk diisi via API yang DILARANG robots.


### Verdict untuk Blibli

| Aspek | Temuan |
|---|---|
| `robots.txt` | API data `/backend/search/*` **DILARANG** (Protego/Scrapy) — sumber data utama halaman |
| Jenis halaman | **SPA** (React) — daftar produk tidak ada di HTML, diisi via API tadi |
| Proteksi | Sering membalas **HTTP 403** ke klien non-browser (anti-bot, perilaku berubah-ubah) |

**Kesimpulan: kita TIDAK men-scrape Blibli.** Data produknya hanya tersedia lewat API yang
**dilarang `robots.txt`**, dan situsnya juga menolak klien non-browser. Mengakali proteksi
anti-bot berisiko melanggar ToS dan keluar dari etika scraping — jadi *Scrapy polos* memang
bukan alat yang tepat di sini.

> Keputusan **berhenti** itu bagian sah dari pekerjaan scraping yang profesional. Bukan gagal —
> justru itu yang membedakan praktisi yang etis.

Lalu bagaimana "real case" yang tetap bisa kita jalankan? Pakai target yang **mengizinkan**.

## 3. Target yang Diizinkan: `books.toscrape.com`

`books.toscrape.com` **tidak punya `robots.txt` yang melarang** (404) dan memang situs publik
untuk latihan. Strukturnya mirip e-commerce nyata sehingga polanya bisa kamu pindahkan ke
toko lain yang mengizinkan:

- **Halaman daftar** → kartu produk + tombol *next* (pagination).
- **Halaman detail** → judul, harga, rating, stok, UPC, kategori.

Kita bangun spider Scrapy "production-style" dengan:

| Komponen | Fungsi |
|---|---|
| `scrapy.Item` | skema data yang rapi |
| `parse` → `parse_detail` | crawl 2 level (daftar → detail) + ikuti pagination |
| `price-parser` | ubah teks harga (`£51.77`) jadi angka `float` |
| `ROBOTSTXT_OBEY=True` | patuhi aturan situs |
| `AUTOTHROTTLE` + `DOWNLOAD_DELAY` | sopan, tidak membanjiri server |
| `CONCURRENT_REQUESTS` | konkurensi (async) yang membuat Scrapy cepat |
| `FEEDS` (`-O file.json`) | ekspor hasil otomatis |

> Polanya identik dengan kalau API Blibli boleh: cukup ganti `start_urls`/parser ke endpoint
> JSON, sisanya (Item, throttling, robots, ekspor) sama.

In [3]:
spider_code = r'''
import scrapy
from price_parser import Price

RATING = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


class BukuItem(scrapy.Item):
    judul     = scrapy.Field()
    kategori  = scrapy.Field()
    harga_teks = scrapy.Field()
    harga     = scrapy.Field()   # float hasil price-parser
    mata_uang = scrapy.Field()
    rating    = scrapy.Field()
    stok      = scrapy.Field()
    upc       = scrapy.Field()
    url       = scrapy.Field()


class BukuSpider(scrapy.Spider):
    name = "buku"
    allowed_domains = ["books.toscrape.com"]
    start_urls = ["https://books.toscrape.com/catalogue/page-1.html"]

    custom_settings = {
        "USER_AGENT": "kelas-scraping-edukasi (+https://github.com/thosangs/dibimbing_scraping)",
        "ROBOTSTXT_OBEY": True,            # patuhi robots.txt
        "CONCURRENT_REQUESTS": 8,           # async: hingga 8 request paralel
        "DOWNLOAD_DELAY": 0.25,             # jeda sopan antar request
        "AUTOTHROTTLE_ENABLED": True,       # sesuaikan kecepatan otomatis
        "AUTOTHROTTLE_TARGET_CONCURRENCY": 4.0,
        "CLOSESPIDER_ITEMCOUNT": 40,        # batasi demo agar ringan & sopan
        "LOG_LEVEL": "INFO",
        "FEED_EXPORT_ENCODING": "utf-8",
    }

    # --- Level 1: halaman daftar -> ikuti tiap produk + pagination ---
    def parse(self, response):
        for card in response.css("article.product_pod"):
            href = card.css("h3 a::attr(href)").get()
            yield response.follow(href, callback=self.parse_detail)

        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)

    # --- Level 2: halaman detail -> ekstrak field lengkap ---
    def parse_detail(self, response):
        teks_harga = response.css("p.price_color::text").get(default="").strip()
        harga = Price.fromstring(teks_harga)

        tabel = {
            row.css("th::text").get(): row.css("td::text").get()
            for row in response.css("table.table-striped tr")
        }
        stok_digit = "".join(c for c in tabel.get("Availability", "") if c.isdigit())
        rating_cls = response.css("p.star-rating::attr(class)").get(default="")
        rating_kata = rating_cls.replace("star-rating", "").strip()

        item = BukuItem()
        item["judul"]      = response.css("div.product_main h1::text").get()
        item["kategori"]   = response.css("ul.breadcrumb li:nth-child(3) a::text").get()
        item["harga_teks"] = teks_harga
        item["harga"]      = harga.amount_float          # price-parser -> float
        item["mata_uang"]  = harga.currency
        item["rating"]     = RATING.get(rating_kata)
        item["stok"]       = int(stok_digit) if stok_digit else 0
        item["upc"]        = tabel.get("UPC")
        item["url"]        = response.url
        yield item
'''

with open("/tmp/buku_spider.py", "w") as f:
    f.write(spider_code)

print("Spider ditulis ke /tmp/buku_spider.py")

Spider ditulis ke /tmp/buku_spider.py


### Jalankan spider

Di Jupyter, Twisted reactor milik Scrapy tidak bisa di-`start` dua kali, jadi kita jalankan
spider lewat **subprocess** (`scrapy runspider`) — sama seperti menjalankannya di terminal /
produksi. Hasilnya diekspor ke JSON via flag `-O`.

In [4]:
import subprocess, sys, os, time

OUT = "/tmp/buku_hasil.json"
if os.path.exists(OUT):
    os.remove(OUT)

t0 = time.time()
res = subprocess.run(
    [sys.executable, "-m", "scrapy", "runspider", "/tmp/buku_spider.py", "-O", OUT],
    capture_output=True, text=True, timeout=180,
)
durasi = time.time() - t0

# tampilkan baris statistik penting dari log Scrapy
kunci = ["robotstxt", "item_scraped_count", "response_received_count",
         "downloader/request_count", "elapsed_time_seconds", "finish_reason"]
for line in res.stderr.splitlines():
    if any(k in line for k in kunci):
        print(line.strip())

print(f"\nReturncode: {res.returncode}  |  Durasi wall-clock: {durasi:.1f}s")

'scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
'downloader/request_count': 55,
'elapsed_time_seconds': 35.000192790990695,
'finish_reason': 'closespider_itemcount',
'item_scraped_count': 47,
'response_received_count': 55,
'robotstxt/request_count': 1,
'robotstxt/response_count': 1,
'robotstxt/response_status_count/404': 1,

Returncode: 0  |  Durasi wall-clock: 36.6s


### Hasil → `pandas` (nyambung ke cleaning & DB)

Output JSON tinggal dibaca ke `DataFrame`. Kolom `harga` sudah `float` (berkat `price-parser`),
jadi siap dianalisis / disimpan ke database seperti di `walkthrough.ipynb`.

In [5]:
import pandas as pd, json

with open("/tmp/buku_hasil.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("Jumlah item :", len(df))
print("\nTipe data:")
print(df.dtypes)

print("\nHarga (GBP):")
print(f"  termurah  : {df['harga'].min():.2f}")
print(f"  termahal  : {df['harga'].max():.2f}")
print(f"  rata-rata : {df['harga'].mean():.2f}")

df[["judul", "kategori", "harga", "mata_uang", "rating", "stok", "upc"]].head(10)

Jumlah item : 47

Tipe data:
judul             str
kategori          str
harga_teks        str
harga         float64
mata_uang         str
rating          int64
stok            int64
upc               str
url               str
dtype: object

Harga (GBP):
  termurah  : 15.94
  termahal  : 57.25
  rata-rata : 35.42


,judul,kategori,harga,mata_uang,rating,stok,upc
0,It's Only the Himalayas,Travel,45.17,£,2,19,a22124811bfa8350
1,Libertarianism for Beginners,Politics,51.33,£,2,19,a18a4f574854aced
2,Mesaerion: The Best Science Fiction Stories 18...,Science Fiction,37.59,£,1,19,e30f54cea9b38190
3,Olio,Poetry,23.88,£,1,19,feb7cc7701ecf901
4,Our Band Could Be Your Life: Scenes from the A...,Music,57.25,£,3,19,deda3e61b9514b83
5,Rip it Up and Start Again,Music,35.02,£,5,19,a34ba96d4081e6a4
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,£,5,19,3b1c02bac2a429e6
7,Set Me Free,Young Adult,17.46,£,5,19,ce6396b0f23f6ecc
8,Behind Closed Doors,Thriller,52.22,£,4,18,be5cc846f45496fb
9,"In a Dark, Dark Wood",Mystery,19.63,£,1,18,19ed25f4641d5efd


## Ringkasan

- **Recon dulu, kode belakangan.** Cek `robots.txt` ("boleh?") dan jenis halaman/proteksi
  ("bisa?") sebelum menulis spider.
- **Blibli → tidak di-scrape**: API-nya `Disallow` di `robots.txt` dan situsnya membalas 403
  (anti-bot). Berhenti adalah keputusan yang benar dan profesional — bukan kegagalan.
- **Scrapy bersinar** untuk crawl multi-level + pagination + konkurensi (async) + ekspor,
  dengan rem etika bawaan: `ROBOTSTXT_OBEY`, `DOWNLOAD_DELAY`, `AUTOTHROTTLE`.
- **Polanya portabel**: spider yang sama (Item, `parse`→`parse_detail`, `price-parser`,
  FEEDS) bisa diarahkan ke toko mana pun yang mengizinkan — atau ke endpoint JSON, cukup ubah
  bagian parser-nya.

### Kalau benar-benar butuh data dari situs ber-proteksi

Tempuh jalur yang sah, bukan menembus paksa:
- Cari **API/partner resmi** atau program afiliasi/feed produk.
- Minta **izin tertulis** ke pemilik situs.
- Gunakan **layanan data berlisensi** (mis. penyedia komersial) yang menanggung kepatuhan.
- Untuk konten yang memang `Allow`, scrape **sopan** (rate-limit, identitas User-Agent jelas).